<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day10-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 10, Segment 3 discussion — contact-map prediction as a *dense* task, and a real parameter contrast

`day10.qmd` mentions `notebooks/protein_contact_prediction.ipynb`'s 2D-CNN contact-map case study without computing anything about it directly. Here: build a small synthetic contact map (same idea, much smaller), run one real 2D convolutional layer over it to confirm the pixel-to-pixel output shape, and compute a real parameter count contrasting a convolutional approach against what a fully-connected model would need for the *same dense, every-pixel-predicts-a-pixel* task — a starker case than Day 10's single-label classification.

In [1]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## 1) A small synthetic contact map

A real contact map is an $L \times L$ grid (L = sequence length) where entry $(i, j)$ marks whether residues $i$ and $j$ are close together in the folded structure. Build a toy $L=30$ version with two synthetic "contact bands" (representing, e.g., two strands of a beta sheet pairing up) to have something concrete to convolve over.

In [2]:
L = 30
contact_map = np.zeros((L, L), dtype=np.float32)

# Band 1: a short helix-like local contact band near the diagonal (i, i+3), typical of an alpha helix's i,i+3/i,i+4 contacts
for i in range(0, L - 3):
    contact_map[i, i + 3] = 1.0
    contact_map[i + 3, i] = 1.0

# Band 2: a longer-range antiparallel contact band (residues 5-12 pairing with 25-18, reversed), typical of a beta strand pair
for offset in range(8):
    contact_map[5 + offset, 25 - offset] = 1.0
    contact_map[25 - offset, 5 + offset] = 1.0

print("contact map shape:", contact_map.shape)
print("fraction of contacts (sparsity, realistic for real contact maps too):", contact_map.mean())

contact map shape: (30, 30)
fraction of contacts (sparsity, realistic for real contact maps too): 0.07777778


## 2) A real 2D convolution over it, confirming pixel-to-pixel output shape

Run one real `nn.Conv2d` layer with `padding="same"`-equivalent padding over the $30\times30$ map and confirm the output is still $30\times30$ — this is the structural property that makes convolution suitable for *dense* prediction (one output value per input pixel), unlike the single-label classifier from earlier in this notebook set.

In [3]:
x = torch.tensor(contact_map).unsqueeze(0).unsqueeze(0)  # (1, 1, 30, 30)

contact_conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
out = contact_conv(x)
print("input shape: ", tuple(x.shape))
print("output shape:", tuple(out.shape))
print("spatial dimensions preserved (dense, pixel-to-pixel):", tuple(x.shape[2:]) == tuple(out.shape[2:]))

input shape:  (1, 1, 30, 30)
output shape: (1, 8, 30, 30)
spatial dimensions preserved (dense, pixel-to-pixel): True


## 3) A real parameter contrast: convolutional vs. fully connected, for a *dense* prediction

A fully-connected model predicting a full $L \times L$ output from a full $L \times L$ input (every output pixel potentially depending on every input pixel) needs $L^2 \times L^2 = L^4$ weights — not just $L^2$. Compute this exactly for $L=30$ (this notebook's toy size) and for a more realistic $L=300$ (a modest real protein), and compare to a small convolutional stack.

In [4]:
def fc_dense_params(L):
    return L**2 * L**2  # L^2 inputs -> L^2 outputs, fully connected

def conv_stack_params(n_layers=4, channels=8, kernel=3, in_channels=1):
    total = 0
    c_in = in_channels
    for _ in range(n_layers):
        total += kernel * kernel * c_in * channels
        c_in = channels
    total += kernel * kernel * c_in * 1  # final 1-channel output layer
    return total

conv_params = conv_stack_params()
print(f"4-layer conv stack (8 channels, 3x3 kernels): {conv_params:,} weights -- independent of L\n")

for L in (30, 300):
    fc_params = fc_dense_params(L)
    print(f"L={L}: fully-connected dense prediction needs {fc_params:,} weights "
          f"({fc_params / conv_params:,.0f}x the conv stack's {conv_params:,})")

4-layer conv stack (8 channels, 3x3 kernels): 1,872 weights -- independent of L

L=30: fully-connected dense prediction needs 810,000 weights (433x the conv stack's 1,872)
L=300: fully-connected dense prediction needs 8,100,000,000 weights (4,326,923x the conv stack's 1,872)


## Discuss

1. The convolutional stack's parameter count doesn't depend on $L$ at all, while the fully-connected count grows as $L^4$. At what real protein length would the fully-connected version need more weights than there are seconds since the Big Bang ($\approx 4.3 \times 10^{17}$)? (Solve $L^4 = 4.3\times10^{17}$ for $L$, then check your answer numerically.)
2. `protein_contact_prediction.ipynb`'s real architecture is a U-Net (an encoder-decoder with skip connections, mentioned on `day10.qmd`), not the plain 4-layer stack used here. Skip connections were also central to ResNet's success (Day 10's classic-architecture section). Why might a *dense*, pixel-to-pixel prediction task specifically benefit from skip connections, beyond just "deeper is better"?

*One group presents.*

In [5]:
# Check your answer to question 1 here.
seconds_since_big_bang = 4.3e17
L_crossover = seconds_since_big_bang ** 0.25
print(f"L^4 = {seconds_since_big_bang:.1e}  =>  L = {L_crossover:.0f}")
print(f"check: {L_crossover:.0f}^4 = {L_crossover**4:.2e}")

L^4 = 4.3e+17  =>  L = 25607
check: 25607^4 = 4.30e+17
